In [ ]:
# Setup for Google Colab
try:
    import google.colab
    IN_COLAB = True
    print("Running in Google Colab")
    
    # Mount Google Drive
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Change to your project directory (update this path to match your Drive structure)
    import os
    os.chdir('/content/drive/MyDrive/EnsembleFederatedLearning')
    
    # Install required packages
    !pip install -q torch torchvision scikit-learn matplotlib seaborn scipy
    
except ImportError:
    IN_COLAB = False
    print("Running locally")

In [ ]:
# CIFAR-10 Rotation-Based Clustering Analysis

This notebook divides CIFAR-10 dataset into 4 rotation-based clusters (0°, 90°, 180°, 270°), assigns them to 50 clients, and performs clustering analysis on the gradients from the second epoch using the last layer.

Running locally


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from torch.utils.data import DataLoader, Subset, Dataset
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA
from scipy.spatial.distance import cosine
import copy
import random
from collections import defaultdict

from training.ensemble_fl import EnsembleFedAvg
from training.utils import get_model

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'data'

In [ ]:
## Configuration

In [ ]:
CONFIG = {
    'num_rotation_clusters': 4,  # 0°, 90°, 180°, 270°
    'num_clients': 50,
    'seed': 42,
    'model_name': 'resnet18',
    'pretrained': False,
    'batch_size': 64,
    'lr': 0.01,
    'warmup_epochs': 2,
}

# Set random seeds
random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed(CONFIG['seed'])

## Step 1: Create Rotation-Based Dataset

We'll create a custom dataset that applies rotations to CIFAR-10 images and organize them into 4 clusters.

In [ ]:
class RotatedCIFAR10Dataset(Dataset):
    """Custom dataset that applies rotation to CIFAR-10 images"""
    def __init__(self, cifar_dataset, rotation_angle, transform=None):
        self.cifar_dataset = cifar_dataset
        self.rotation_angle = rotation_angle
        self.transform = transform
        self.base_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010))
        ])
        
    def __len__(self):
        return len(self.cifar_dataset)
    
    def __getitem__(self, idx):
        image, label = self.cifar_dataset[idx]
        
        # Apply rotation
        if self.rotation_angle != 0:
            image = transforms.functional.rotate(image, self.rotation_angle)
        
        # Apply transform
        image = self.base_transform(image)
        
        return image, label

# Load CIFAR-10 dataset
print("Loading CIFAR-10 dataset...")
train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")

## Step 2: Create 4 Rotation Clusters and Assign to 50 Clients

We'll divide the dataset into 4 rotation-based clusters and then distribute these to 50 clients.

In [ ]:
# Create 4 rotation clusters
rotation_angles = [0, 90, 180, 270]
rotation_clusters = {}

for angle in rotation_angles:
    rotation_clusters[angle] = RotatedCIFAR10Dataset(train_dataset, angle)
    print(f"Cluster {angle}°: {len(rotation_clusters[angle])} samples")

# Assign rotation clusters to 50 clients
# Each client will get data from one rotation cluster
# We'll distribute clients evenly across the 4 clusters (12-13 clients per cluster)

clients_per_cluster = CONFIG['num_clients'] // CONFIG['num_rotation_clusters']
remainder = CONFIG['num_clients'] % CONFIG['num_rotation_clusters']

client_assignments = []  # List of (rotation_angle, client_data_indices)
client_rotation_labels = []  # Track which rotation cluster each client belongs to

for cluster_idx, angle in enumerate(rotation_angles):
    # Number of clients for this cluster
    n_clients_this_cluster = clients_per_cluster + (1 if cluster_idx < remainder else 0)
    
    # Get total samples in this rotation cluster
    total_samples = len(rotation_clusters[angle])
    samples_per_client = total_samples // n_clients_this_cluster
    
    # Shuffle indices for random distribution
    indices = list(range(total_samples))
    random.shuffle(indices)
    
    # Assign data to clients
    for i in range(n_clients_this_cluster):
        start_idx = i * samples_per_client
        end_idx = start_idx + samples_per_client if i < n_clients_this_cluster - 1 else total_samples
        client_indices = indices[start_idx:end_idx]
        
        client_assignments.append((angle, client_indices))
        client_rotation_labels.append(angle)

print(f"\nTotal clients created: {len(client_assignments)}")
print(f"Clients per rotation cluster: {[client_rotation_labels.count(a) for a in rotation_angles]}")

In [ ]:
# Create client subsets
train_subsets = []
for angle, indices in client_assignments:
    dataset = rotation_clusters[angle]
    subset = Subset(dataset, indices)
    train_subsets.append(subset)

print(f"Created {len(train_subsets)} client subsets")
print(f"Sample sizes: min={min(len(s) for s in train_subsets)}, max={max(len(s) for s in train_subsets)}")

## Step 3: Visualize Data Distribution

Let's visualize how the data is distributed across clients and rotation clusters.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Client distribution across rotation clusters
rotation_counts = [client_rotation_labels.count(a) for a in rotation_angles]
axes[0].bar([f"{a}°" for a in rotation_angles], rotation_counts, color=['blue', 'green', 'orange', 'red'])
axes[0].set_xlabel('Rotation Angle')
axes[0].set_ylabel('Number of Clients')
axes[0].set_title('Client Distribution Across Rotation Clusters')
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Data samples per client
client_sizes = [len(subset) for subset in train_subsets]
axes[1].hist(client_sizes, bins=20, edgecolor='black', alpha=0.7)
axes[1].set_xlabel('Number of Samples')
axes[1].set_ylabel('Number of Clients')
axes[1].set_title('Distribution of Data Samples per Client')
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.show()

print(f"\nStatistics:")
print(f"Average samples per client: {np.mean(client_sizes):.1f}")
print(f"Std dev: {np.std(client_sizes):.1f}")

## Step 4: Initialize Model and Run Warmup

We'll use the EnsembleFedAvg class with warmup training for 2 epochs to collect gradients.

In [ ]:
# Prepare test dataset
test_rotated = RotatedCIFAR10Dataset(test_dataset, 0)  # Use 0° for test set
test_subset = Subset(test_rotated, list(range(len(test_dataset))))

# Initialize EnsembleFedAvg
ensemble_fl = EnsembleFedAvg(
    train_subsets=train_subsets,
    test_set=test_subset,
    num_clients=CONFIG['num_clients'],
    device=device,
    model_name=CONFIG['model_name'],
    pretrained=CONFIG['pretrained'],
    batch_size=CONFIG['batch_size'],
    lr=CONFIG['lr'],
    seed=CONFIG['seed']
)

print("EnsembleFedAvg initialized successfully")

In [ ]:
# Run warmup training for 2 epochs
print("Starting warmup training...")
ensemble_fl.run_warmup(use_fedavg=False, local_epochs=CONFIG['warmup_epochs'])
print("Warmup training complete!")

## Step 5: Extract Gradients from Last Layer (Second Epoch)

Extract the gradients from the last layer (fc layer) for each client from the second epoch.

In [ ]:
# Extract gradients from the last layer (fc layer) for the second epoch (epoch index 1)
epoch_idx = 1  # Second epoch (0-indexed)

# Find the last layer name (typically 'fc.weight' and 'fc.bias' for ResNet)
sample_client = 0
if sample_client in ensemble_fl.warmup_layer_gradients:
    sample_layer_grads = ensemble_fl.warmup_layer_gradients[sample_client][0]
    layer_names = list(sample_layer_grads.keys())
    print(f"Available layers: {layer_names[-5:]}")  # Show last 5 layers
    
    # Find last layer (fc layer)
    conv_layers = [name for name in layer_names if 'fc' in name or 'layer4' in name]
    print(f"FC layers: {conv_layers}")

# Extract gradients from fc.weight and fc.bias
last_layer_gradients = []
for client_idx in range(CONFIG['num_clients']):
    if client_idx in ensemble_fl.warmup_layer_gradients and len(ensemble_fl.warmup_layer_gradients[client_idx]) > epoch_idx:
        epoch_grads = ensemble_fl.warmup_layer_gradients[client_idx][epoch_idx]
        
        # Concatenate fc.weight and fc.bias gradients
        fc_grads = []
        for layer_name in fc_layers:
            if layer_name in epoch_grads:
                fc_grads.append(epoch_grads[layer_name].flatten())
        
        if fc_grads:
            combined_fc_grad = np.concatenate(fc_grads)
            last_layer_gradients.append(combined_fc_grad)
        else:
            print(f"Warning: No FC gradients found for client {client_idx}")
    else:
        print(f"Warning: Client {client_idx} has no gradients for epoch {epoch_idx}")

last_layer_gradients = np.array(last_layer_gradients)
print(f"\nExtracted gradients shape: {last_layer_gradients.shape}")
print(f"Number of clients with gradients: {len(last_layer_gradients)}")

## Step 6: Perform Clustering Analysis

We'll perform K-Means clustering on the gradients and analyze how well it aligns with the rotation-based clusters.

In [ ]:
# Perform K-Means clustering with k=4 (matching rotation clusters)
n_clusters = 4
kmeans = KMeans(n_clusters=n_clusters, random_state=CONFIG['seed'], n_init=10)
predicted_clusters = kmeans.fit_predict(last_layer_gradients)

# Calculate clustering metrics
silhouette = silhouette_score(last_layer_gradients, predicted_clusters)

print(f"K-Means Clustering Results (k={n_clusters}):")
print(f"Silhouette Score: {silhouette:.4f}")
print(f"\nCluster sizes:")
for i in range(n_clusters):
    count = np.sum(predicted_clusters == i)
    print(f"  Cluster {i}: {count} clients")

## Step 7: Compare Predicted Clusters with True Rotation Clusters

Analyze how well the gradient-based clustering aligns with the actual rotation-based clusters.

In [ ]:
# Map rotation angles to cluster IDs
rotation_to_id = {0: 0, 90: 1, 180: 2, 270: 3}
true_clusters = np.array([rotation_to_id[angle] for angle in client_rotation_labels])

# Create confusion matrix
confusion_matrix = np.zeros((n_clusters, n_clusters), dtype=int)
for true_label, pred_label in zip(true_clusters, predicted_clusters):
    confusion_matrix[true_label, pred_label] += 1

# Plot confusion matrix
plt.figure(figsize=(10, 8))
sns.heatmap(confusion_matrix, annot=True, fmt='d', cmap='Blues', 
            xticklabels=[f'Pred {i}' for i in range(n_clusters)],
            yticklabels=[f'True {i} ({rotation_angles[i]}°)' for i in range(n_clusters)])
plt.title('Confusion Matrix: True Rotation Clusters vs Predicted Clusters')
plt.ylabel('True Rotation Cluster')
plt.xlabel('Predicted Cluster')
plt.tight_layout()
plt.show()

# Calculate purity for each cluster
print("\nCluster Purity Analysis:")
for pred_cluster in range(n_clusters):
    mask = predicted_clusters == pred_cluster
    if np.sum(mask) > 0:
        true_labels_in_cluster = true_clusters[mask]
        unique, counts = np.unique(true_labels_in_cluster, return_counts=True)
        dominant_label = unique[np.argmax(counts)]
        purity = np.max(counts) / np.sum(mask)
        print(f"  Predicted Cluster {pred_cluster}:")
        print(f"    - Dominant rotation: {rotation_angles[dominant_label]}°")
        print(f"    - Purity: {purity:.2%}")
        print(f"    - Distribution: {dict(zip([rotation_angles[u] for u in unique], counts))}")

## Step 8: Visualize Gradients with PCA

Use PCA to reduce dimensionality and visualize the gradient space.

In [ ]:
# Apply PCA to reduce to 2D for visualization
pca = PCA(n_components=2, random_state=CONFIG['seed'])
gradients_2d = pca.fit_transform(last_layer_gradients)

print(f"PCA Explained Variance Ratio: {pca.explained_variance_ratio_}")
print(f"Total variance explained: {sum(pca.explained_variance_ratio_):.2%}")

# Create visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Colored by true rotation clusters
colors_true = ['blue', 'green', 'orange', 'red']
for i, angle in enumerate(rotation_angles):
    mask = true_clusters == i
    axes[0].scatter(gradients_2d[mask, 0], gradients_2d[mask, 1], 
                    c=colors_true[i], label=f'{angle}°', alpha=0.6, s=100)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[0].set_title('Gradient Space - True Rotation Clusters')
axes[0].legend()
axes[0].grid(alpha=0.3)

# Plot 2: Colored by predicted clusters
colors_pred = plt.cm.tab10(np.linspace(0, 1, n_clusters))
for i in range(n_clusters):
    mask = predicted_clusters == i
    axes[1].scatter(gradients_2d[mask, 0], gradients_2d[mask, 1], 
                    c=[colors_pred[i]], label=f'Cluster {i}', alpha=0.6, s=100)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)')
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)')
axes[1].set_title('Gradient Space - Predicted Clusters (K-Means)')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## Step 9: Gradient Similarity Analysis

Analyze the similarity between gradients using cosine similarity.

In [ ]:
# Compute cosine similarity matrix
from sklearn.metrics.pairwise import cosine_similarity

similarity_matrix = cosine_similarity(last_layer_gradients)

# Calculate average within-cluster and between-cluster similarities
within_cluster_sims = []
between_cluster_sims = []

for i in range(CONFIG['num_clients']):
    for j in range(i+1, CONFIG['num_clients']):
        sim = similarity_matrix[i, j]
        if true_clusters[i] == true_clusters[j]:
            within_cluster_sims.append(sim)
        else:
            between_cluster_sims.append(sim)

print("Gradient Similarity Analysis:")
print(f"  Within-cluster similarity: {np.mean(within_cluster_sims):.4f} ± {np.std(within_cluster_sims):.4f}")
print(f"  Between-cluster similarity: {np.mean(between_cluster_sims):.4f} ± {np.std(between_cluster_sims):.4f}")
print(f"  Difference: {np.mean(within_cluster_sims) - np.mean(between_cluster_sims):.4f}")

# Plot similarity matrices
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Sort clients by rotation cluster for better visualization
sorted_indices = np.argsort(true_clusters)
sorted_similarity = similarity_matrix[sorted_indices][:, sorted_indices]

# Plot 1: Full similarity matrix (sorted by true clusters)
im1 = axes[0].imshow(sorted_similarity, cmap='RdYlGn', vmin=-1, vmax=1)
axes[0].set_title('Cosine Similarity Matrix (sorted by rotation cluster)')
axes[0].set_xlabel('Client Index')
axes[0].set_ylabel('Client Index')
plt.colorbar(im1, ax=axes[0])

# Add lines to separate rotation clusters
boundaries = [0]
for angle in rotation_angles[:-1]:
    count = np.sum(true_clusters == rotation_to_id[angle])
    boundaries.append(boundaries[-1] + count)
for boundary in boundaries[1:]:
    axes[0].axhline(y=boundary, color='black', linewidth=2)
    axes[0].axvline(x=boundary, color='black', linewidth=2)

# Plot 2: Average similarity per rotation cluster pair
avg_sim_matrix = np.zeros((4, 4))
for i in range(4):
    for j in range(4):
        mask_i = true_clusters == i
        mask_j = true_clusters == j
        cluster_sims = similarity_matrix[np.ix_(mask_i, mask_j)]
        avg_sim_matrix[i, j] = np.mean(cluster_sims)

im2 = axes[1].imshow(avg_sim_matrix, cmap='RdYlGn', vmin=-1, vmax=1)
axes[1].set_title('Average Similarity Between Rotation Clusters')
axes[1].set_xticks(range(4))
axes[1].set_yticks(range(4))
axes[1].set_xticklabels([f'{a}°' for a in rotation_angles])
axes[1].set_yticklabels([f'{a}°' for a in rotation_angles])
axes[1].set_xlabel('Rotation Cluster')
axes[1].set_ylabel('Rotation Cluster')
plt.colorbar(im2, ax=axes[1])

# Add text annotations
for i in range(4):
    for j in range(4):
        text = axes[1].text(j, i, f'{avg_sim_matrix[i, j]:.3f}',
                           ha="center", va="center", color="black", fontsize=10)

plt.tight_layout()
plt.show()

## Step 10: Additional Clustering Algorithms

Compare K-Means with other clustering algorithms (Hierarchical, Spectral).

In [ ]:
from sklearn.cluster import SpectralClustering

# Try different clustering algorithms
clustering_results = {}

# K-Means (already computed)
clustering_results['K-Means'] = predicted_clusters

# Hierarchical Clustering
hierarchical = AgglomerativeClustering(n_clusters=n_clusters, linkage='ward')
clustering_results['Hierarchical'] = hierarchical.fit_predict(last_layer_gradients)

# Spectral Clustering
spectral = SpectralClustering(n_clusters=n_clusters, random_state=CONFIG['seed'], affinity='rbf')
clustering_results['Spectral'] = spectral.fit_predict(last_layer_gradients)

# Calculate metrics for each algorithm
print("Comparison of Clustering Algorithms:\n")
for method_name, predictions in clustering_results.items():
    silhouette = silhouette_score(last_layer_gradients, predictions)
    
    # Calculate accuracy by finding best mapping
    from scipy.optimize import linear_sum_assignment
    confusion = np.zeros((n_clusters, n_clusters), dtype=int)
    for true_label, pred_label in zip(true_clusters, predictions):
        confusion[true_label, pred_label] += 1
    
    row_ind, col_ind = linear_sum_assignment(-confusion)
    accuracy = confusion[row_ind, col_ind].sum() / len(true_clusters)
    
    print(f"{method_name}:")
    print(f"  Silhouette Score: {silhouette:.4f}")
    print(f"  Clustering Accuracy: {accuracy:.2%}")
    print()

## Step 11: Summary and Conclusions

Let's summarize the key findings from this analysis.

In [ ]:
print("=" * 80)
print("CLUSTERING ANALYSIS SUMMARY")
print("=" * 80)
print(f"\nDataset Configuration:")
print(f"  - Total clients: {CONFIG['num_clients']}")
print(f"  - Rotation clusters: {CONFIG['num_rotation_clusters']} (0°, 90°, 180°, 270°)")
print(f"  - Training epochs: {CONFIG['warmup_epochs']}")
print(f"  - Model: {CONFIG['model_name']}")

print(f"\nData Distribution:")
print(f"  - Clients per rotation cluster: {[client_rotation_labels.count(a) for a in rotation_angles]}")
print(f"  - Average samples per client: {np.mean([len(s) for s in train_subsets]):.0f}")

print(f"\nGradient Analysis:")
print(f"  - Gradient dimension (last layer): {last_layer_gradients.shape[1]}")
print(f"  - Epoch used: 2 (index 1)")

print(f"\nClustering Performance:")
best_method = max(clustering_results.items(), 
                  key=lambda x: silhouette_score(last_layer_gradients, x[1]))
print(f"  - Best method: {best_method[0]}")
print(f"  - Silhouette score: {silhouette_score(last_layer_gradients, best_method[1]):.4f}")

print(f"\nKey Insights:")
print(f"  1. Within-cluster similarity: {np.mean(within_cluster_sims):.4f}")
print(f"  2. Between-cluster similarity: {np.mean(between_cluster_sims):.4f}")
print(f"  3. Separation: {np.mean(within_cluster_sims) - np.mean(between_cluster_sims):.4f}")
print(f"  4. The gradient-based clustering {'successfully' if np.mean(within_cluster_sims) - np.mean(between_cluster_sims) > 0.1 else 'partially'} ")
print(f"     identifies the rotation-based data heterogeneity")
print("=" * 80)